In [6]:
import torch.nn as nn
import torch
import pandas as pd

In [7]:
data = pd.read_csv("Pokemon.csv")
data.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


## Descripción de la tarea

Trabajaremos con un dataset de la serie "Pokémon" que desglosa las capacidades de cada criatura en atributos cuantitativos. La tarea consiste en utilizar arquitecturas de redes neuronales simples (MLP) para identificar patrones que distingan a los Pokémon Legendarios del resto. Deberán procesar estas variables y entrenar un clasificador que maximice la capacidad predictiva sobre la variable objetivo "Legendary".

Consideraciones:
- Debe entregarlo a más tardar el 29 de mayo a las 18:00 horas.
- Debe ser entregado al correo luis.llanca@uach.cl con el asunto "Tarea-1-MLP", el archivo debe llamarse NG-MLP-Tarea1.ipynb donde NG es el número de grupo. Es importante que el asunto sea exactamente el mismo. También, se les pedirá que se anoten en la plantilla (que se compartirá posteriormente) para una pequeña interrogación.
- Por cada 20 minutos de retraso, se descontará una décima de la nota. 
- Si necesitan ayuda, pueden escribir a los correos luis.llanca@uach.cl, luis.llanca@cenia.cl o escribir al discord de usuario: llanking (tengo una foto de mi gata). PD: prefiero mucho más el discord. 

# Explicación del Dataset

El dataset utilizado contiene información de distintos Pokémon y sus estadísticas base. Cada fila representa un Pokémon y cada columna corresponde a una característica específica utilizada para analizar o clasificar al Pokémon.

La variable objetivo del problema es `Legendary`, por lo que se trata de un problema de clasificación binaria utilizando redes neuronales, tal como se explica en el material del curso de Redes Neuronales y Machine Learning de Pablo Huijse.

---

## Descripción de las columnas

| Columna | Tipo de dato | Descripción |
|---|---|---|
| `#` | Entero | Número identificador del Pokémon en la Pokédex. |
| `Name` | Texto | Nombre del Pokémon. |
| `Type 1` | Texto | Tipo principal del Pokémon. |
| `Type 2` | Texto / Nulo | Tipo secundario del Pokémon. Algunos no poseen segundo tipo. |
| `Total` | Entero | Suma total de las estadísticas base. |
| `HP` | Entero | Cantidad de puntos de vida. |
| `Attack` | Entero | Nivel de ataque físico. |
| `Defense` | Entero | Nivel de defensa física. |
| `Sp. Atk` | Entero | Poder de ataque especial. |
| `Sp. Def` | Entero | Defensa contra ataques especiales. |
| `Speed` | Entero | Velocidad del Pokémon. |
| `Generation` | Entero | Generación a la que pertenece el Pokémon. |
| `Legendary` | Booleano | Indica si el Pokémon es legendario (`True`) o no (`False`). |

---

## Información relevante del dataset

- Las columnas de estadísticas (`HP`, `Attack`, `Defense`, etc.) corresponden a variables numéricas utilizadas como entrada del modelo.

- Las columnas `Type 1` y `Type 2` son variables categóricas, por lo que deben transformarse a formato numérico antes de entrenar una red neuronal.

- La columna `Legendary` corresponde a la variable objetivo del problema, por lo que el modelo debe aprender a clasificar entre Pokémon legendarios y no legendarios.

- Existen valores faltantes en `Type 2`, ya que algunos Pokémon tienen solamente un tipo.

- Según el material del curso, es recomendable normalizar las variables numéricas antes del entrenamiento para mejorar el funcionamiento del gradiente descendente y del perceptrón multicapa (MLP).

- El dataset presenta desbalance de clases, debido a que existen muchos menos Pokémon legendarios que normales. El material del curso recomienda considerar métricas adicionales al accuracy y utilizar particiones estratificadas en estos casos.




# Preparación del dataset

Antes de entrenar el modelo fue necesario realizar una preparación exhaustiva de los datos para asegurar un entrenamiento robusto y evitar filtraciones de información (*data leakage*):

* Se eliminaron las columnas `Name`, `#` y `Generation`, ya que no aportaban información relevante para la clasificación. En el caso de `#`, existían valores repetidos debido a las Mega-Evoluciones; `Name` correspondía a texto libre, y `Generation` no presentaba una relación directa con la identificación de Pokémon legendarios.
* La variable objetivo `Legendary` (booleana) se transformó a formato numérico binario (1 y 0) para el cálculo de la función de pérdida.
* Las variables categóricas `Type 1` y `Type 2` fueron transformadas mediante **One-Hot Encoding**, incluyendo una categoría explícita para los valores nulos de `Type 2` para evitar pérdida de datos, debido a que las redes neuronales trabajan con entradas puramente numéricas.
* Se realizó una **partición estratificada** de los datos en conjuntos de entrenamiento y prueba. Esto es necesario debido al desbalance de clases del dataset, garantizando que la baja proporción de Pokémon Legendarios se mantenga constante en ambos conjuntos.
* Las variables numéricas continuas fueron normalizadas utilizando **StandardScaler** para que todas se encontraran en escalas similares, lo que ayuda a la convergencia del modelo mediante gradiente descendente. El ajuste del escalador se realizó exclusivamente sobre el conjunto de entrenamiento para no contaminar el modelo con la distribución estadística de los datos de prueba.
* Los conjuntos resultantes se transformaron a **Tensores flotantes de PyTorch** (`torch.float32`), cumpliendo con la estructura de datos obligatoria para alimentar las capas densas del modelo.

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Cargar y limpiar columnas no relevantes
df = pd.read_csv("Pokemon.csv")
df = df.drop(columns=["Name", "#", "Generation"])

# 2. Separar características (X) y objetivo (y)
X = df.drop(columns=["Legendary"])
y = df["Legendary"].astype(int) # Convertimos True/False a 1 y 0

# 3. Aplicar One-Hot Encoding a variables categóricas
X = pd.get_dummies(X, columns=["Type 1", "Type 2"], dummy_na=True) # dummy_na maneja los nulos de Type 2

# 4. Dividir los datos en Entrenamiento (Train) y Prueba (Test) de forma estratificada
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 5. Escalar SOLO las variables numéricas continuas
num_cols = ['Total', 'HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']
scaler = StandardScaler()

# Ajustamos el escalador solo en Train y lo aplicamos a Train y Test
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 6. Convertir a Tensores de PyTorch para el MLP
# Convertimos los booleanos del One-Hot a enteros antes de pasarlos a tensores flotantes
X_train_tensor = torch.tensor(X_train.astype(float).values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1) 

X_test_tensor = torch.tensor(X_test.astype(float).values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

## Definición del modelo

Para resolver el problema de clasificación binaria se definieron tres arquitecturas distintas de Perceptrón Multicapa (MLP) en el archivo `models.py`. Cada arquitectura posee una complejidad diferente, permitiendo comparar capacidad de aprendizaje y generalización.

---

### Arquitectura 1: `MLP_Small`

```python
class MLP_Small(nn.Module):
```

Esta arquitectura corresponde al modelo más simple utilizado en la tarea. Posee una única capa oculta de 16 neuronas y utiliza la función de activación ReLU.

Su objetivo principal fue servir como modelo base para comparar rendimiento y capacidad de aprendizaje frente a arquitecturas más complejas.

Al tener pocos parámetros, el riesgo de sobreajuste es menor, aunque también posee una capacidad limitada para aprender relaciones complejas dentro del dataset.

---

### Arquitectura 2: `MLP_Medium`

```python
class MLP_Medium(nn.Module):
```

Esta arquitectura incorpora dos capas ocultas, permitiendo aumentar la capacidad de representación del modelo.

La reducción progresiva de neuronas (`32 -> 16`) busca extraer características de manera gradual y aprender relaciones no lineales más complejas entre las estadísticas de los Pokémon y la variable objetivo `Legendary`.

Esta arquitectura representa un equilibrio entre complejidad y capacidad de generalización.

---

### Arquitectura 3: `MLP_Large`

```python
class MLP_Large(nn.Module):
```

Esta arquitectura corresponde al modelo más complejo utilizado. Posee una capa oculta más amplia y utiliza regularización mediante `Dropout`.

El uso de `Dropout(p=0.3)` permite desactivar neuronas aleatoriamente durante el entrenamiento, ayudando a reducir el sobreajuste y mejorando la capacidad de generalización del modelo.

Según el material del curso, cuando aumenta la cantidad de parámetros de una red neuronal también aumenta el riesgo de sobreajuste, por lo que el uso de regularización resulta importante en arquitecturas más grandes.

---

### Selección de la mejor arquitectura

La arquitectura seleccionada para el entrenamiento final fue `MLP_Large`, debido a que presentó el mejor desempeño en validación y una mejor capacidad de generalización respecto a las demás arquitecturas.

Además, el uso de Dropout permitió controlar el sobreajuste durante el entrenamiento, manteniendo un equilibrio adecuado entre complejidad y rendimiento.

La selección final se realizó comparando métricas de entrenamiento y validación, siguiendo las recomendaciones del material del curso sobre evaluación y generalización de modelos.

In [11]:
from sklearn.model_selection import train_test_split

# División entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [12]:
from models import MLP_Large

# Crear modelo
model = MLP_Large(input_dim=X_train.shape[1])

## Definición de optimizador y función de costo

Para el entrenamiento del Perceptrón Multicapa (MLP) se utilizó el optimizador **Adam** junto con la función de costo **Binary Cross-Entropy**, implementada en PyTorch mediante `nn.BCEWithLogitsLoss()`.

#### Justificación

La función de costo `BCEWithLogitsLoss` fue seleccionada porque el problema corresponde a una clasificación binaria, donde el modelo debe predecir si un Pokémon es legendario (`1`) o no (`0`). Esta función permite medir qué tan diferentes son las predicciones del modelo respecto a las etiquetas reales.

Además, esta implementación combina internamente la función sigmoide con la entropía cruzada binaria, lo que mejora la estabilidad numérica durante el entrenamiento y evita problemas en el cálculo de gradientes.

Por otro lado, se utilizó el optimizador Adam debido a que es un método basado en gradiente descendente adaptativo. Según el material del curso, Adam ajusta automáticamente la tasa de aprendizaje para cada parámetro de la red utilizando información de los gradientes y de los gradientes cuadrados.

Esto permite que el entrenamiento sea más rápido, estable y eficiente en comparación con métodos tradicionales como SGD, especialmente en redes neuronales multicapa.

Finalmente, se utilizó una tasa de aprendizaje (`learning rate`) de `0.001`, ya que corresponde a un valor comúnmente utilizado para lograr una convergencia estable durante el entrenamiento.

In [14]:
import torch.nn as nn
import torch.optim as optim

# Función de costo
criterion = nn.BCEWithLogitsLoss()

# Optimizador
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Entrenamiento del modelo

El modelo seleccionado (`MLP_Large`) fue entrenado utilizando el dataset previamente preparado y normalizado.

Para el entrenamiento se utilizó un tamaño de batch (`batch_size`) de 32 muestras, permitiendo actualizar los pesos de la red neuronal de manera más eficiente y estable durante el proceso de gradiente descendente.

Se entrenó el modelo durante 50 épocas (`epochs`), realizando múltiples iteraciones completas sobre el conjunto de entrenamiento para permitir que la red aprendiera patrones presentes en los datos.

Durante cada época se realizaron las siguientes etapas:

- propagación hacia adelante (*forward pass*),
- cálculo de la función de pérdida,
- cálculo de gradientes mediante *backpropagation*,
- actualización de pesos utilizando el optimizador Adam.

El optimizador Adam fue utilizado debido a que adapta automáticamente la tasa de aprendizaje para cada parámetro de la red, mejorando la estabilidad y velocidad de convergencia del entrenamiento.

La función de costo utilizada fue `BCEWithLogitsLoss`, adecuada para problemas de clasificación binaria como la predicción de Pokémon legendarios.

Además, los datos fueron cargados mediante `DataLoader`, permitiendo trabajar con batches y mezclar aleatoriamente los datos de entrenamiento (`shuffle=True`), ayudando a mejorar la capacidad de generalización del modelo.

Durante el entrenamiento se observó una disminución progresiva de la función de pérdida a lo largo de las épocas, pasando de valores cercanos a `0.59` hasta aproximadamente `0.01`.

Esto indica que el modelo logró aprender correctamente patrones presentes en el dataset y que el proceso de optimización mediante Adam y backpropagation convergió de manera estable.

La selección de hiperparámetros se realizó siguiendo las recomendaciones del material del curso relacionadas con gradiente descendente, optimización adaptativa, monitoreo de pérdida y control del sobreajuste.

In [15]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convertir datos a tensores
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# Crear datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# DataLoader
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size
)

# Número de épocas
epochs = 50

# Entrenamiento
for epoch in range(epochs):

    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:

        # Reiniciar gradientes
        optimizer.zero_grad()

        # Forward
        outputs = model(inputs)

        # Calcular pérdida
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Actualizar pesos
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")

Epoch [1/50] - Loss: 0.5942
Epoch [2/50] - Loss: 0.4228
Epoch [3/50] - Loss: 0.2831
Epoch [4/50] - Loss: 0.2113
Epoch [5/50] - Loss: 0.1827
Epoch [6/50] - Loss: 0.1578
Epoch [7/50] - Loss: 0.1434
Epoch [8/50] - Loss: 0.1301
Epoch [9/50] - Loss: 0.1135
Epoch [10/50] - Loss: 0.0986
Epoch [11/50] - Loss: 0.0858
Epoch [12/50] - Loss: 0.0736
Epoch [13/50] - Loss: 0.0726
Epoch [14/50] - Loss: 0.0607
Epoch [15/50] - Loss: 0.0557
Epoch [16/50] - Loss: 0.0524
Epoch [17/50] - Loss: 0.0480
Epoch [18/50] - Loss: 0.0459
Epoch [19/50] - Loss: 0.0439
Epoch [20/50] - Loss: 0.0377
Epoch [21/50] - Loss: 0.0409
Epoch [22/50] - Loss: 0.0327
Epoch [23/50] - Loss: 0.0341
Epoch [24/50] - Loss: 0.0303
Epoch [25/50] - Loss: 0.0243
Epoch [26/50] - Loss: 0.0296
Epoch [27/50] - Loss: 0.0207
Epoch [28/50] - Loss: 0.0295
Epoch [29/50] - Loss: 0.0252
Epoch [30/50] - Loss: 0.0226
Epoch [31/50] - Loss: 0.0183
Epoch [32/50] - Loss: 0.0210
Epoch [33/50] - Loss: 0.0222
Epoch [34/50] - Loss: 0.0231
Epoch [35/50] - Loss: 0

### Evaluación del modelo
Evalúe el modelo utilizando métricas adecuadas para este problema de clasificación. Justifique la selección de las métricas utilizadas y discuta los resultados obtenidos. 

### Preguntas finales
1. Sobre la matriz de confusión, interprete los resultados obtenidos. Con sus palabras defina que significa cada tipo de error. ¿Elegiría a Pokémon ubicados en FP o FN para su equipo?
2. Busque un caso mal clasificado por el modelo, e interprete por qué cree que el modelo se equivocó en ese caso.
3. ¿Cúal fue el mayor desafío que enfrentó al realizar esta tarea? ¿Cómo lo solucionó?



### IA Generativa

**1. ¿Utilizó alguna herramienta de IA Generativa para realizar esta tarea? En caso afirmativo, indique cuál o cuáles herramientas utilizó.**

Sí, se utilizó IA generativa a lo largo de gran parte del trabajo, principalmente como herramienta de apoyo y ayuda. Las herramientas utilizadas fueron Gemini, ChatGPT y Copilot.

---

**2. ¿En qué parte o partes de la tarea utilizó estas herramientas?**

La IA fue utilizada en distintas partes del trabajo, específicamente para:

- Identificar en qué documentos y secciones se encontraba la información necesaria para responder preguntas o realizar actividades.
- Aclarar conceptos y términos relacionados con redes neuronales y machine learning.
- Explicar fragmentos de código y apoyar en su comprensión.
- Detectar y solucionar errores de código cuyo origen no era claro.
- Mejorar la redacción y organización del texto del informe.